# D76 消融A: 关掉选路锚(schedule KL)——`abl_noroute_s0`

与 d73_AB **唯一差异** = `--kl-anchor none --kl-beta 0`(时机锚保留)。补齐「双锚」故事: 时机锚已有干净 on/off(D70), 本 run 补选路锚的。预期: 选路漂移 → 延误升/方差大; 若居然不差, 也是诚实结论(后期选路已内化)。

## 设置

In [ ]:
import sys, os
venv_site = '/rds/homes/h/hxn886/virtual-environments/my-virtual-env-sapphire/lib/python3.11/site-packages'
if os.path.isdir(venv_site) and venv_site not in sys.path:
    sys.path.insert(0, venv_site)
%cd /rds/homes/h/hxn886/ondemand/final_project
PROJ='/rds/homes/h/hxn886/ondemand/final_project'
PY='/rds/homes/h/hxn886/virtual-environments/my-virtual-env-sapphire/bin/python'
PREFIX=(f'cd {PROJ} && env PYTHONUNBUFFERED=1 PYTHONPATH=. '
        f'PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True MPLBACKEND=Agg {PY}')
print('OK')


## 训练(50 轮, 与 d73_AB 前 50 轮同预算对比)

In [ ]:
!{PREFIX} scripts/13_train_ppo.py \
    --run-tag abl_noroute_s0 \
    --kl-anchor none --kl-beta 0 \
    --curriculum --curriculum-tiers 185,286,300,999999 --promote-at 1.0 --promote-delay-k 1.0 \
    --feature-version 2 --seed 0 \
    --init-ckpt artifacts/reports/ppo_push_AB_s0_best.pt --anchor-ckpt artifacts/reports/ppo_push_AB_s0_best.pt \
    --bc-anchor-beta 0.01 --lr 5e-5 \
    --encoder gnn --control-classes 12 --reserve-window wide \
    --train-eval-days 6 --iters 50 --steps-per-iter 8192 --eval-every 5 \
    --early-stop-patience 8 --best-rescue --resume


## 对比(vs 完整版前 50 轮)

In [ ]:
import json
runs=[('本消融','abl_noroute_s0'),('完整版 d73_AB(前50轮)','ppo_d73_AB_s0')]
for name,tag in runs:
    try: d=json.load(open(f'artifacts/reports/{tag}.json'))
    except FileNotFoundError:
        import torch; d=torch.load(f'artifacts/reports/{tag}_last.pt',map_location='cpu',weights_only=False)
    evs=[e for e in d['evals'] if e['iter']<50]
    b=min(evs,key=lambda e:e['eval_weighted_secondary_delay_s'])
    print(f"{name:<22} 前50轮best: 完成 {b['eval_completion_rate']:.3f} | 延误 {b['eval_weighted_secondary_delay_s']:>9.0f} | reward {b['eval_episode_reward']:>8.0f} @it{b['iter']}")
    for e in evs: print(f"   it{e['iter']:>3} 档{e.get('tier',0)+1} {e['eval_weighted_secondary_delay_s']:>9.0f}")
